# Introduction to the Claude API

**Live online course — instructor walkthrough notebook**

This notebook follows the course lecture notes. Each section has:
- **Lecture notes** (markdown) — what to explain on the slide/screen.
- **Demo code** — cells to run live for students to see real output.
- **🏫 During class** callouts — specific instructor actions, questions to ask, and variations to try.

---

## Agenda

1. Overview of Claude Models
2. Accessing the API (how a request actually flows)
3. Making a Request
4. Multi-Turn Conversations
5. System Prompts
6. Temperature
7. Response Streaming
8. Controlling Model Output (pre-fill + stop sequences)
9. Structured Data Generation
10. Recap + practice exercises


## 0. Setup (do this before class starts)

1. Install dependencies:
   ```bash
   pip install anthropic python-dotenv
   ```
2. Create a file named `.env` in the same directory as this notebook containing:
   ```
   ANTHROPIC_API_KEY="sk-ant-...your-key..."
   ```
3. Add `.env` to `.gitignore` so it is never committed to version control.

> **🏫 During class:** Before running the first cell, open the `.env` file and show students what it looks like (blur the key). Emphasize: **never paste the API key directly into a notebook cell** — `.env` + `python-dotenv` keeps secrets out of source control.


In [1]:
# Install packages (uncomment if not already installed)
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


1. load_dotenv() — reads the .env file sitting next to the notebook and copies every KEY="value" line into the process's environment variables. After this runs, os.getenv("ANTHROPIC_API_KEY") returns the secret you stored on line 8 of your .env file. The key never appears in the notebook itself, so it can't leak via screenshare or git commits.                                                                             
2. client = anthropic.Anthropic() — creates the SDK client. We pass no arguments on purpose: the SDK automatically looks for ANTHROPIC_API_KEY in the environment (which load_dotenv() just populated). This client object is what you'll call .messages.create(...) on for the rest of the course.                                                                                                                                   
3. model = "claude-sonnet-4-6" — a named constant so that every later cell refers to model instead of hard-coding the string. Swap this one line and every demo in the notebook switches model.
4. The three print(...) lines — a sanity check. If class is about to start and something's wrong, these tell you immediately:                                                                                      
    - SDK version → confirms the anthropic package installed correctly.                                                                                                                                              
    - Model → confirms the name you'll use in requests.                                                                                                                                                              
    - Key loaded: True → confirms the .env was found and read. If this prints False, the .env file isn't in the same directory as the notebook (or is named env.txt, etc.) — fix it before running any API calls.  

In [2]:
from dotenv import load_dotenv
import anthropic
import os

load_dotenv()  # loads ANTHROPIC_API_KEY from .env

client = anthropic.Anthropic()  # picks up ANTHROPIC_API_KEY automatically

# We'll use Sonnet 4.6 as the default workhorse model for the course.
model = "claude-sonnet-4-6"

print("SDK version:", anthropic.__version__)
print("Model:", model)
print("Key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))

SDK version: 0.96.0
Model: claude-sonnet-4-6
Key loaded: True


---
# 1. Overview of Claude Models

Claude has three model **families**, each optimized for a different priority:

| Family | Strength | Trade-off | Good for |
|---|---|---|---|
| **Opus** | Highest intelligence; deep reasoning + planning | Higher cost & latency | Complex, multi-step tasks |
| **Sonnet** | Balanced intelligence, speed, cost; strong coding + precise edits | — | **Most practical use cases** |
| **Haiku** | Fastest & cheapest | No extended reasoning | Real-time UX, high-volume processing |

### Selection framework
- Intelligence priority → **Opus**
- Speed priority → **Haiku**
- Balanced → **Sonnet**

### Common pattern
Real apps often use **multiple models in the same application**, routing each task to the right model. Example: use Haiku to classify incoming tickets, then Opus to draft the hard ones.

All three share the core capabilities — text generation, coding, image analysis. The difference is **optimization focus**.

> **🏫 During class:** Ask the room: *"If you're building a customer support autoresponder that handles 10,000 messages/day, which model?"* then *"What if you're building a contract analyzer that runs on 50 docs per week?"* Use this to reinforce the selection framework.


 This cell runs a side-by-side comparison of the three Claude model families on one identical prompt. The flow:                                                                                                     
                                                                                                                                                                                                                     
  1. question — a single fixed prompt ("Explain recursion to a 10-year-old in one sentence."). Using the same input for every model is what makes the comparison fair.                                               
  2. for m in [...] — loops over the three model names, from fastest/cheapest (claude-haiku-4-5) to most capable/expensive (claude-opus-4-7).                                                                        
  3. start = time.time() → elapsed = time.time() - start — wraps each API call so we can print how many seconds each model took.                                                                                     
  4. client.messages.create(...) — the actual API call. Only model=m changes from one iteration to the next; max_tokens, messages, and the prompt are identical.                                                     
  5. The two print lines — show a header with the model name + elapsed time, then the generated answer from resp.content[0].text.                 

In [3]:
# Quick sanity check: same question, three models, different tone/length/latency
import time

question = "Explain recursion to a 10-year-old in one sentence."

for m in ["claude-haiku-4-5", "claude-sonnet-4-6", "claude-opus-4-7"]:
    start = time.time()
    resp = client.messages.create(
        model=m,
        max_tokens=200,
        messages=[{"role": "user", "content": question}],
    )
    elapsed = time.time() - start
    print(f"--- {m}  ({elapsed:.2f}s) ---")
    print(resp.content[0].text)
    print()

--- claude-haiku-4-5  (0.88s) ---
Recursion is when a function calls itself to solve a problem by breaking it into smaller and smaller pieces until it reaches a piece so small it's easy to solve.

--- claude-sonnet-4-6  (1.45s) ---
Recursion is when something solves a big problem by solving a smaller version of the same problem, over and over, until it's simple enough to just answer.

--- claude-opus-4-7  (2.03s) ---
Recursion is when something solves a problem by doing a smaller version of the same thing over and over — like looking up "recursion" in a dictionary and the definition says "see recursion."



---
# 2. Accessing the API

A Claude-powered app has **5 steps** from user input to response display.

```
[ User / Client ]  --(1)-->  [ Your Server ]  --(2)-->  [ Anthropic API ]
                                                             |
                                                            (3) text generation
                                                             |
                                                            (4) stop
                                                             |
[ User / Client ]  <--(5)--  [ Your Server ]  <----------  [ Anthropic API ]
```

**Step 1 — Client → your server.** The user types text in the UI; the client sends it to *your* server. Never call the Anthropic API directly from the client (your API key would leak).

**Step 2 — Server → Anthropic.** Your server makes the API request using an SDK (Python, TypeScript/JavaScript, Go, Ruby) or plain HTTP. Required pieces: **API key**, **model name**, **messages list**, **max_tokens**.

**Step 3 — Text generation inside the model.** Four internal stages:
1. **Tokenization** — break input into tokens (words / word parts / symbols / spaces).
2. **Embedding** — each token → a list of numbers representing all its possible meanings.
3. **Contextualization** — adjust each embedding based on neighboring tokens to fix its precise meaning in *this* sentence.
4. **Generation** — output layer produces a probability distribution over possible next tokens; pick one (probability + randomness); append; repeat.

**Step 4 — Stop.** Generation stops when either:
- `max_tokens` is reached, or
- the model emits a special **end-of-sequence** token.

**Step 5 — Response back.** API returns the generated text + usage counts + `stop_reason` to your server, which forwards to the client.

### Glossary
- **Token** — chunk of text (word / part / symbol).
- **Embedding** — numerical representation of word meanings.
- **Contextualization** — meaning refinement using neighboring words.
- **max_tokens** — generation length *limit*.
- **stop_reason** — why the model stopped generating.

> **🏫 During class:** Draw the 5-step diagram on the whiteboard/virtual board and narrate each hop. Then ask: *"Why can't the browser call Anthropic directly?"* — key answer: **the API key is a secret**.


---
# 3. Making a Request

### Request structure
```python
client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[{"role": "user", "content": "What is quantum computing?"}],
)
```

**Required arguments:**
- `model` — name of the Claude model.
- `max_tokens` — **safety limit** on generation length, not a target.
- `messages` — a list of conversation exchanges.

### Message types
- **User message:** `{"role": "user", "content": "..."}` — human-authored.
- **Assistant message:** model-generated response.

### Accessing the response
- `message` — the full response object (includes metadata + usage + nested content).
- `message.content[0].text` — just the generated text.


 1. client.messages.create(...) — the single function that sends a request to Claude. Three required arguments:                                                                                                     
    - model=model — the model constant we pinned in the setup cell (claude-sonnet-4-6).                                                                                                                              
    - max_tokens=1000 — a safety cap on how long the reply can get. Not a target length, not padding — just an upper bound.                                                                                          
    - messages=[{"role": "user", "content": "..."}] — a list of one message: a user turn containing the question "What is quantum computing? Answer in 3 sentences."                                                 
  2. message = ... — the call returns a Message object (the IDE hover tooltip in the screenshot even confirms the type: message: Message). This object holds everything the API sent back, not just the text.        
  3. The print block unpacks that object in three layers so students see what's actually inside:                                                                                                                     
    - print(message) — the full response object: id, model, role, content, stop_reason, stop_sequence, usage, etc.                                                                                                   
    - message.content[0].text — just the generated text, the piece you'd actually show in a UI.                                                                                                                      
    - message.stop_reason — why generation stopped (typically "end_turn", or "max_tokens" if the cap was hit).                                                                                                       
    - message.usage — token counts (input_tokens, output_tokens) — what you'd log for billing and quota tracking.      

In [4]:
# First real request
message = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=[{"role": "user", "content": "What is quantum computing? Answer in 3 sentences."}],
)

print("--- Full response object ---")
print(message)
print()
print("--- Just the text ---")
print(message.content[0].text)
print()
print("stop_reason :", message.stop_reason)
print("usage       :", message.usage)

--- Full response object ---
Message(id='msg_01AnrcnAVhmTDG6XmQieiYDo', container=None, content=[TextBlock(citations=None, text='Quantum computing is a type of computation that harnesses quantum mechanical phenomena, such as superposition and entanglement, to process information in fundamentally different ways than classical computers. Unlike classical bits, which exist as either 0 or 1, quantum bits (qubits) can exist in multiple states simultaneously, allowing quantum computers to explore many possible solutions at once. This makes quantum computers potentially far more powerful than classical computers for specific tasks, such as cryptography, drug discovery, and optimization problems.', type='text')], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inf

> **🏫 During class:**
> 1. Run the cell above. Show students the **full object** first — point out `id`, `model`, `stop_reason`, `usage.input_tokens`, `usage.output_tokens`.
> 2. Then zoom in on `message.content[0].text` — the piece you'd actually put in a UI.
> 3. Ask the class to **change the prompt** and re-run — this is their first live interaction.


---
# 4. Multi-Turn Conversations

A **multi-turn conversation** is a back-and-forth that maintains context across exchanges.

### Key limitation
**The Anthropic API stores nothing between calls.** Every request is independent. The model has no memory of what it said a minute ago.

### Solution
1. **You** maintain the message list in your code.
2. **You** send the *entire* history with every follow-up request.

### Flow
```
send initial user msg → get assistant reply
append assistant reply to history
append next user msg to history
send full history again → get contextual reply
... repeat ...
```

We'll build three small helpers and then demonstrate the difference between sending history vs. not.


In [5]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """Send messages to Claude and return the assistant text."""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        # Sonnet 4.6 / Opus 4.7 default to adaptive extended thinking, which is
        # mutually exclusive with assistant-message pre-fill (used in §8a + §9).
        # Disable it here so every demo in the notebook behaves consistently.
        "thinking": {"type": "disabled"},
    }
    if system is not None:
        params["system"] = system
    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences

    response = client.messages.create(**params)
    return response.content[0].text

In [6]:
# A. WITHOUT history — each call is independent, so context is lost
msgs = []
add_user_message(msgs, "My name is Priya and I love jazz music.")
reply1 = chat(msgs)
print("Assistant 1:", reply1, "\n")

# Start a FRESH list (simulating forgetting history)
msgs_fresh = []
add_user_message(msgs_fresh, "What is my name and what kind of music do I like?")
reply2 = chat(msgs_fresh)
print("Assistant 2 (no history):", reply2)

Assistant 1: Hi Priya! It's great to meet you! Jazz is such a rich and diverse genre. Do you have a particular style or era of jazz you're most drawn to? For example:

- **Classic/Traditional** jazz from the early 1900s
- **Bebop** (Miles Davis, Charlie Parker)
- **Cool Jazz** or **Hard Bop**
- **Fusion** or **Contemporary jazz**

Or maybe a favorite artist or album? I'd love to chat about it! 🎷 

Assistant 2 (no history): I don't have any information about you personally. I don't know your name or your music preferences. I can only know what you share with me in our conversation.

Would you like to tell me about yourself? 😊


In [7]:
# B. WITH history — Claude has full context
msgs = []
add_user_message(msgs, "My name is Priya and I love jazz music.")
reply1 = chat(msgs)
add_assistant_message(msgs, reply1)

add_user_message(msgs, "What is my name and what kind of music do I like?")
reply2 = chat(msgs)
add_assistant_message(msgs, reply2)

print("Assistant (with history):", reply2)
print()
print("--- Full message history we're maintaining ---")
for m in msgs:
    print(f"[{m['role']}] {m['content']}")

Assistant (with history): Based on what you told me earlier, your name is **Priya** and you love **jazz music**! 🎷

--- Full message history we're maintaining ---
[user] My name is Priya and I love jazz music.
[assistant] Hi Priya! It's great to meet you! Jazz is a wonderful genre - so rich and diverse. Do you have any particular style of jazz you enjoy most, like:

- **Classic bebop** (Charlie Parker, Dizzy Gillespie)
- **Cool jazz** (Miles Davis, Chet Baker)
- **Smooth jazz**
- **Fusion**
- **Vocal jazz** (Ella Fitzgerald, Billie Holiday)

Or do you have favorite artists or albums you'd recommend? I'd love to chat about it! 🎷
[user] What is my name and what kind of music do I like?
[assistant] Based on what you told me earlier, your name is **Priya** and you love **jazz music**! 🎷


> **🏫 During class:**
> 1. Run cell A first — show that the second call doesn't know the name.
> 2. Run cell B — same question now gets a correct contextual answer.
> 3. **Key takeaway to say out loud:** *"The API is stateless. Memory is your job."*
> 4. Ask a student to add a 3rd turn (e.g., *"Recommend me an album in that genre."*) and watch context carry.


---
# 5. System Prompts

A **system prompt** customizes Claude's style, tone, and behavior by assigning it a role.

- Passed as a plain string via the `system=` keyword.
- Controls **how** Claude answers, not **what** it answers.
- Typical structure: first line assigns a role, then behavioral instructions.

**Example:** A *math tutor* system prompt makes Claude offer hints instead of giving away the answer — even though the user's question is unchanged.


In [8]:
question = "What's the derivative of x^3 + 2x?"

# --- Without a system prompt ---
msgs = []
add_user_message(msgs, question)
print("--- No system prompt ---")
print(chat(msgs))
print()

# --- With a tutor system prompt ---
tutor_system = (
    "You are a patient math tutor for high school students. "
    "When a student asks a question, never give the final answer directly. "
    "Instead, ask a leading question or give a small hint that guides them "
    "to work the next step out themselves."
)
msgs = []
add_user_message(msgs, question)
print("--- Tutor system prompt ---")
print(chat(msgs, system=tutor_system))

--- No system prompt ---
## Derivative of x³ + 2x

Using basic differentiation rules:

$$f(x) = x^3 + 2x$$

$$f'(x) = 3x^2 + 2$$

**Steps:**
- **x³** → bring down the exponent, reduce it by 1: **3x²**
- **2x** → the derivative of a linear term is just its coefficient: **2**

--- Tutor system prompt ---
Great question! Let's work through this together.

Do you remember the **Power Rule** for taking derivatives? It says that for a term like xⁿ, the derivative is n·xⁿ⁻¹.

Can you try applying that rule to the **first term, x³**? What do you get?


> **🏫 During class:**
> 1. Run both variants side-by-side and ask the class: *"Same question, two totally different answers — what changed?"*
> 2. Ask a student to propose a new role (e.g., *"pirate sommelier"*, *"terse senior engineer on PR review"*), swap it in, re-run.
> 3. Emphasize: **system prompts steer behavior; the user message provides the task.**


---
# 6. Temperature

**Temperature** is a parameter (0–1) that controls randomness in token selection.

### How it works
Text generation assigns a probability to every possible next token. Temperature reshapes that distribution:
- **Temperature 0** → deterministic; always picks the highest-probability token.
- **Higher temperature** → flattens the distribution; lower-probability tokens have a better chance of being picked → more creative / unexpected output.

### When to use what
| Task | Temperature |
|---|---|
| Data extraction, classification, factual answers | Near **0** |
| Brainstorming, creative writing, jokes, marketing copy | Near **1** |

Higher values **don't guarantee** different outputs — they just raise the probability of variation.


In [9]:
prompt = "Write one short, unusual tagline for a coffee brand aimed at night-owl programmers."

for temp in [0.0, 0.0, 1.0, 1.0]:
    msgs = [{"role": "user", "content": prompt}]
    reply = chat(msgs, temperature=temp)
    print(f"temperature={temp} → {reply}")

temperature=0.0 → **"Compile at midnight. Ship at dawn."**
temperature=0.0 → **"Compile at midnight. Ship at dawn."**
temperature=1.0 → **"Compile at 2AM. Ship at dawn."**
temperature=1.0 → **"Compile at midnight. Ship by dawn."**


> **🏫 During class:**
> 1. Run the cell. Point out: the two `temperature=0.0` runs are (nearly) identical. The two `temperature=1.0` runs usually differ.
> 2. Ask: *"If you were building a tax-form extractor, which temperature? A Twitter joke generator?"*
> 3. Variation: swap the prompt for `"Extract the invoice number from: INV-2026-00423"` and show that even at temp=1 the answer barely varies — because the factual distribution is already sharply peaked.


---
# 7. Response Streaming

**Streaming** displays the response chunk-by-chunk as it's generated instead of waiting for the full response.

### Why it matters
A long response can take 10–30 seconds. Users expect *immediate* feedback — a loading spinner feels broken. Streaming gives them live typing.

### How it works
1. Server sends user message to Claude.
2. Claude sends an initial acknowledgement event (no text yet).
3. A stream of **events** follows, each carrying text deltas.
4. Your server forwards deltas to the frontend to render in real time.

### Key event types
| Event | Meaning |
|---|---|
| `message_start` | Initial acknowledgement |
| `content_block_start` | A text block begins |
| `content_block_delta` | **Actual text chunk** (the one you render) |
| `content_block_stop` / `message_stop` | Generation complete |

### Two ways to stream in the SDK
- **Low-level:** `client.messages.create(stream=True)` returns an iterator of raw events.
- **High-level:** `client.messages.stream(...)` gives you a `text_stream` that yields only the text chunks, plus `get_final_message()` to assemble the complete message for storage.


In [10]:
# Low-level: iterate raw events. Useful when you want full control.
import sys

with client.messages.stream(
    model=model,
    max_tokens=300,
    messages=[{"role": "user", "content": "Tell me a 4-sentence bedtime story about a curious otter."}],
) as stream:
    for text_chunk in stream.text_stream:
        print(text_chunk, end="", flush=True)

    # After the stream completes, grab the full assembled message
    final = stream.get_final_message()

print("\n\n--- Final assembled message metadata ---")
print("stop_reason :", final.stop_reason)
print("usage       :", final.usage)

Here is a 4-sentence bedtime story about a curious otter:

Luna the little otter spent all day exploring the riverbank, poking her whiskered nose into every hollow log and mossy stone she could find. By evening, she had discovered a shimmering dragonfly, three smooth pebbles, and a patch of the softest mud she had ever felt between her paws. As the stars began to blink awake above the water, Luna floated on her back and held her favorite pebble on her belly, just like a little treasure. With the gentle river rocking her like a lullaby, her curious eyes slowly closed, and she drifted off to dream of all the wonderful things she would discover tomorrow.

--- Final assembled message metadata ---
stop_reason : end_turn
usage       : Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=23, output_tokens=162, server_tool_use=None, service_tier='standard')


> **🏫 During class:**
> 1. Before running, contrast with an earlier non-streaming cell: *"Remember how the cell paused, then dumped 200 words at once? Watch this one instead."*
> 2. Run the streaming cell — students see text type out live.
> 3. Point out `get_final_message()` — you still get the complete message for database storage, token counts, etc.
> 4. Key UX principle to say: *"Streaming doesn't make the model faster. It makes the **wait** feel faster."*


---
# 8. Controlling Model Output

Beyond rewriting the prompt, two techniques give you precise control over the response.

## 8a. Pre-filling Assistant Messages

Append an **assistant** message to the end of the `messages` list — Claude treats that text as something it already started saying, and continues from exactly where you left off.

- Steers the response direction.
- Claude continues **from the exact endpoint** of the pre-fill — not from the start of a new sentence.
- You must stitch the pre-fill + generated continuation yourself if you want the whole thing.


In [11]:
# Pre-fill: steer Claude toward a specific stance
messages = [
    {"role": "user", "content": "Is coffee or tea the better morning drink?"},
    {"role": "assistant", "content": "Coffee is better because"},  # ← pre-fill
]

continuation = chat(messages)
print("Pre-fill + continuation:")
print("Coffee is better because" + continuation)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'This model does not support assistant message prefill. The conversation must end with a user message.'}, 'request_id': 'req_011CaHkKRh9MsrVxkk4s8tvy'}

## 8b. Stop Sequences

A **stop sequence** is a string that, once Claude generates it, causes generation to halt immediately. The triggering string itself is **not** included in the output.

- Set via `stop_sequences=[...]`.
- Great for trimming output to a boundary (end of a section, end of a list, closing fence).

Two quick examples:


In [ ]:
# Example 1: stop at the word "five"
msgs = [{"role": "user", "content": "Count from one to ten, separated by commas."}]
print("Stop on 'five':")
print(chat(msgs, stop_sequences=["five"]))
print()

# Example 2: refined stop sequence for a cleaner cut
print("Stop on ', five':")
print(chat(msgs, stop_sequences=[", five"]))

Stop on 'five':
One, two, three, four, 

Stop on ', five':
One, two, three, four


> **🏫 During class:**
> 1. Run the pre-fill example — point out how `"Coffee is better because"` is already committed in the messages, and Claude just picks up where it left off.
> 2. Ask: *"Notice we had to stitch the pre-fill back in for display — why?"* (Claude only returns the continuation.)
> 3. Run the stop-sequence examples. Show that *"five"* is never in the output.
> 4. Quick takeaway: *"Pre-fill steers direction. Stop sequences control length/boundary. Both work without changing the user prompt."*


---
# 9. Structured Data Generation

By default, Claude wraps structured output (JSON, code, etc.) in markdown fences and often adds explanatory commentary. For programmatic use you usually want **just the raw structured data**.

### The pattern — combine pre-fill + stop sequence
1. User message: request for structured data.
2. Assistant pre-fill: the **opening delimiter** (e.g., ` ```json `).
3. Stop sequence: the **closing delimiter** (e.g., ` ``` `).

Claude sees the opening fence as already written, emits only the inner content, and stops at the closing fence. You get clean, directly-parseable output.

Works for any structured format — JSON, Python code, CSV, YAML — just swap the delimiters.


In [ ]:
import json

user_request = (
    "Give me a JSON array of 3 fictional customer support tickets. "
    "Each ticket has: id (int), customer_name (string), priority (low|medium|high), "
    "and summary (string)."
)

messages = [
    {"role": "user", "content": user_request},
    # Pre-fill must NOT end with trailing whitespace — the API rejects it.
    # So we open the fence without the newline; Claude will continue from here.
    {"role": "assistant", "content": "```json"},
]

raw = chat(messages, stop_sequences=["```"])  # stop at the closing fence

print("--- Raw output ---")
print(raw)

print("\n--- Parsed as Python ---")
tickets = json.loads(raw)  # json.loads tolerates leading whitespace/newline
for t in tickets:
    print(t)

BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'This model does not support assistant message prefill. The conversation must end with a user message.'}, 'request_id': 'req_011CaHizV1RieZdRYgnxqePH'}

> **🏫 During class:**
> 1. Run the cell. Show that `raw` is parseable directly — **no manual string cleanup**.
> 2. Point at the pre-fill line and say: *"The API rejects any assistant pre-fill that ends with whitespace — including a trailing `\n`. That's why the fence is `` ```json `` with no newline. Claude emits its own newline when it continues, and `json.loads()` tolerates leading whitespace."* (Want to show the error live? Append `\n` to the pre-fill and re-run — you'll see `BadRequestError: 'final assistant content cannot end with trailing whitespace'`.)
> 3. Ask: *"If I removed the pre-fill and stop sequence, what would break?"* — answer: `json.loads()` would fail because of markdown fences and any surrounding commentary.
> 4. Variation to try live: swap the delimiters for Python code (\`\`\`python … \`\`\`) to generate a runnable snippet. Tie it together: *"Structured output is just pre-fill + stop sequence applied with intent."*

---
# 10. Recap + practice exercises

### Recap (run through these out loud)
- **Models:** Opus = smart, Sonnet = balanced, Haiku = fast. Mix them per task.
- **Flow:** client → your server → Anthropic → your server → client.
- **Request basics:** `model`, `max_tokens`, `messages`.
- **Memory is yours to maintain** — append to a list and resend it every turn.
- **System prompt** steers *how*; user message supplies *what*.
- **Temperature:** low for facts, high for creativity.
- **Streaming:** improves perceived latency, doesn't change actual latency.
- **Pre-fill + stop sequences** = precise control without rewriting the prompt.
- **Structured output** = pre-fill the opening fence, stop on the closing fence.

### Exercises (do the first in class, assign the rest)
1. **Chat loop:** Build a `while True:` loop that reads `input()`, appends to history, prints Claude's reply, and keeps the conversation going. (Exit on `"quit"`.)
2. **Persona swap:** Take your chat loop and let the user type `/persona <description>` to swap the system prompt live.
3. **Deterministic extractor:** Given a block of text, extract `{name, email, phone}` as JSON using `temperature=0`, pre-fill, and a stop sequence.
4. **Streaming UI feel:** Rewrite exercise 1 using `client.messages.stream()` so replies type out live.
5. **Model router:** Write a function that sends short factual questions to Haiku and long reasoning questions to Opus based on prompt length or a simple classifier call.


In [ ]:
# Exercise 1 scaffold — finish this live in class together.
# Uncomment to run interactively in Jupyter.
#
# conversation = []
# system = "You are a friendly, concise assistant."
# while True:
#     user_text = input("You: ")
#     if user_text.strip().lower() == "quit":
#         break
#     add_user_message(conversation, user_text)
#     reply = chat(conversation, system=system)
#     add_assistant_message(conversation, reply)
#     print(f"Claude: {reply}\n")